# 쇼핑몰에서 누가 자주 오고 많이 사는지 분석해서 단골 찾기

H&M 패션 쇼핑몰의 거래·고객·상품 데이터(상품 이미지 포함)를 분석한다. 이 노트북은 `src/pipeline.py`의 `DataAnalyzer` 클래스를 불러와 다음 순서로 진행한다.

1. 데이터 로드와 기본 탐색
2. 피처 엔지니어링(상품명, 이미지)
3. 결측치 대치와 이상치 처리
4. 통계 분석과 시각화
5. RFM 고객 세분화

In [1]:
import sys
from pathlib import Path

# 이 노트북은 notebooks/ 폴더에서 실행되므로, 부모 폴더(프로젝트 루트)를
# import 경로에 추가해 src/pipeline.py를 불러올 수 있게 한다.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import DataAnalyzer

## 1. 데이터 로드

`DataAnalyzer().load_data()`는 거래·고객·상품 CSV 세 개를 읽는다. 이때 두 가지를 지킨다.

- `customer_id`, `article_id` 같은 ID 컬럼은 **문자열**로 읽는다. 숫자로 읽으면 `0706016001` 같은 상품 ID의 앞자리 0이 사라져 이미지 경로와 다른 테이블과의 연결이 끊긴다.
- `order_date`, `first_sold_date`는 **날짜형**으로 읽는다.

In [2]:
analyzer = DataAnalyzer().load_data()

tables = {
    "transactions (거래)": analyzer.transactions,
    "customers (고객)": analyzer.customers,
    "articles (상품)": analyzer.articles,
}
for name, df in tables.items():
    print(f"{name:<22} {df.shape[0]:>6,}행 x {df.shape[1]}열")

transactions (거래)      27,728행 x 5열
customers (고객)          2,000행 x 7열
articles (상품)           1,500행 x 29열


In [3]:
print("article_id 예시:", analyzer.articles["article_id"].iloc[0])
print("article_id 자료형:", analyzer.articles["article_id"].dtype)

order_dates = analyzer.transactions["order_date"]
print("order_date 자료형:", order_dates.dtype)
print("order_date 범위:", order_dates.min().date(), "~", order_dates.max().date())

article_id 예시: 0706016001
article_id 자료형: str
order_date 자료형: datetime64[us]
order_date 범위: 2019-09-23 ~ 2020-09-22
